# Attach Execution Price To LSTM Dataset

This notebook aligns the cleaned price data back onto the exact rows used by `LSTM_DATA.parquet`.

Join keys:
- `slug`
- `timestamp`

Output:
- `execution_price`
- merged dataset saved to `../data/LSTM_DATA_WITH_PRICE.parquet`


In [ ]:
import pandas as pd
import numpy as np

LSTM_PATH = "../data/LSTM_DATA.parquet"
PRICE_PATH = "../data/Final_cleaned_data.parquet"
OUTPUT_PATH = "../data/LSTM_DATA_WITH_PRICE.parquet"

PRICE_CANDIDATES = [
    "execution_price",
    "Up_Mid_Price",
    "price",
    "Up_trade_close",
    "Up_micro_price",
]


In [ ]:
df_lstm = pd.read_parquet(LSTM_PATH)
df_price = pd.read_parquet(PRICE_PATH)

print("df_lstm:", df_lstm.shape)
print("df_price:", df_price.shape)
print("df_lstm columns:", df_lstm.columns.tolist())
print("df_price columns:", df_price.columns.tolist())


In [ ]:
required_key_cols = {"slug", "timestamp"}
if not required_key_cols.issubset(df_lstm.columns):
    raise ValueError(f"df_lstm is missing required keys: {required_key_cols - set(df_lstm.columns)}")
if not required_key_cols.issubset(df_price.columns):
    raise ValueError(f"df_price is missing required keys: {required_key_cols - set(df_price.columns)}")

price_col = next((col for col in PRICE_CANDIDATES if col in df_price.columns), None)
if price_col is None:
    raise ValueError(
        "Could not find a price column in df_price. "
        f"Checked: {PRICE_CANDIDATES}"
    )

print("Using price column:", price_col)

df_lstm = df_lstm.copy()
df_price = df_price.copy()

df_lstm["timestamp"] = pd.to_datetime(df_lstm["timestamp"])
df_price["timestamp"] = pd.to_datetime(df_price["timestamp"])

df_lstm = df_lstm.sort_values(["slug", "timestamp"]).reset_index(drop=True)
df_price = df_price.sort_values(["slug", "timestamp"]).reset_index(drop=True)

price_key_counts = df_price.groupby(["slug", "timestamp"]).size()
duplicate_price_keys = price_key_counts[price_key_counts > 1]
if not duplicate_price_keys.empty:
    print("Warning: duplicate keys found in df_price. Keeping first row per slug/timestamp.")
    print(duplicate_price_keys.head())
    df_price = df_price.drop_duplicates(subset=["slug", "timestamp"], keep="first").copy()

lstm_key_counts = df_lstm.groupby(["slug", "timestamp"]).size()
duplicate_lstm_keys = lstm_key_counts[lstm_key_counts > 1]
if not duplicate_lstm_keys.empty:
    print("Warning: duplicate keys found in df_lstm. The merge will preserve row count, but duplicates exist on the LSTM side.")
    print(duplicate_lstm_keys.head())


In [ ]:
price_lookup = (
    df_price[["slug", "timestamp", price_col]]
    .rename(columns={price_col: "execution_price"})
    .copy()
)

df_lstm_with_price = df_lstm.merge(
    price_lookup,
    on=["slug", "timestamp"],
    how="left",
    validate="many_to_one",
)

match_mask = df_lstm_with_price["execution_price"].notna()
match_rate = match_mask.mean() if len(df_lstm_with_price) > 0 else np.nan

summary = pd.DataFrame(
    [
        {
            "lstm_rows": len(df_lstm),
            "price_rows": len(df_price),
            "merged_rows": len(df_lstm_with_price),
            "matched_rows": int(match_mask.sum()),
            "unmatched_rows": int((~match_mask).sum()),
            "match_rate": float(match_rate) if pd.notna(match_rate) else np.nan,
            "execution_price_source": price_col,
        }
    ]
)

display(summary)
display(df_lstm_with_price.head())

if len(df_lstm_with_price) != len(df_lstm):
    raise ValueError("Row count changed after merge, which should not happen for a left join.")


In [ ]:
unmatched_examples = df_lstm_with_price.loc[
    df_lstm_with_price["execution_price"].isna(),
    ["slug", "timestamp"]
].head(20)

print("Unmatched examples:")
display(unmatched_examples)

df_lstm_with_price.to_parquet(OUTPUT_PATH, index=False)
print(f"Saved merged dataset to: {OUTPUT_PATH}")
